# Notebook 17: ML Model Watermarking — Embedding Ownership Proofs

## Why This Matters

We've seen that model stealing (notebook 12) can extract a functionally equivalent surrogate from a deployed API. The uncomfortable reality is that *preventing* extraction is largely impossible — given enough queries, an adversary can always build a knockoff. So the question shifts from prevention to detection: can you *prove* that a suspected model was stolen from yours?

This is exactly what ML watermarking provides. A watermark is a secret, verifiable pattern embedded in a model's behavior or weights. If the watermark appears in a third-party model, it's forensic evidence of theft — provable in court.

The analogy to traditional digital watermarking (steganography in images, audio) is imperfect. ML models are trained, not encoded. A watermark that survives model stealing and fine-tuning without degrading the model's clean performance, and that a sophisticated adversary cannot remove, is a hard technical challenge. Several commercial companies (Dioptra, Protect AI, Arthur AI) now offer watermarking as part of their model security stack.

## Background

### Backdoor-Based Watermarking

Adi et al. (2018) and Zhang et al. (2018) independently proposed the key insight: use backdoor-style techniques as a *benign* watermark. The model owner trains the model with a secret set of (trigger_image, target_label) pairs — inputs that look random or nonsensical to others but produce a specific output that only the owner knows to expect. If the model is stolen, these trigger inputs still work because the surrogate learned (via distillation) to mimic the victim's full output distribution, including the watermark behavior.

This is the watermark scheme we previewed in notebook 12. Here we develop it fully:
- **Unrelated images** (e.g., random noise, logos): triggers visually unrelated to the task
- **Pattern-based** (e.g., specific pixel patterns): more subtle, harder to detect
- **Content-based**: meaning-bearing images with unexpected labels (e.g., image of a cat → predicted as class 7)

### Weights-Based Watermarking

Uchida et al. (2017) proposed embedding the watermark *in the model weights* rather than in the input-output behavior. They use a regularization term during training that biases the weights toward a secret binary pattern. Verification: project the weights onto the secret key vector — a watermarked model produces a recognizable pattern; a random model produces noise.

This approach is more resistant to black-box extraction (since it requires weight access to verify) but requires retraining from scratch with the regularizer.

### Robustness Requirements

A practical watermark must survive:
1. **Model stealing** (extraction via knowledge distillation): the surrogate must inherit the watermark
2. **Fine-tuning**: the owner's model might be fine-tuned on a downstream task — the watermark should persist
3. **Pruning**: unimportant weights can be zeroed without affecting accuracy; the watermark must be in the "important" weights
4. **Adversarial removal**: if the adversary knows a watermark exists and tries to remove it by fine-tuning on the trigger inputs with random labels

No watermark scheme is known to survive all of these simultaneously. There's a fundamental tension between watermark strength (embedding deeply in the model) and clean performance (the watermark should not degrade accuracy).

### Statistical Hypothesis Testing for Verification

When verifying a suspected knockoff, you can't just say "the watermark appears." You need a *p-value*: if the model had no watermark, what's the probability it would produce the observed trigger-response pattern by chance? For a 10-class classifier with k trigger inputs, the probability of hitting the right target class by chance is 0.1^k. With 20 trigger inputs all predicting the correct target, the p-value is 10^{-20} — essentially conclusive.

## Structure of This Notebook

1. Train baseline (unwatermarked) model
2. Implement backdoor-style watermarking (content-based triggers)
3. Embed weights-based watermark via regularization
4. Verify watermarks with statistical hypothesis test
5. Robustness tests: fine-tuning, pruning, model stealing
6. Watermark removal attack and its limitations

## What You'll Learn

- How backdoor-style watermarks embed ownership proofs in model behavior
- Why surrogate models inherit watermarks through knowledge distillation
- How to statistically verify a watermark (p-value framework)
- How fine-tuning and pruning affect watermark persistence
- The fundamental robustness-utility tradeoff in watermarking

In [ ]:
%pip install torch torchvision numpy matplotlib scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset, ConcatDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import copy

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Model and Data Setup

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)


def train_model(model, train_loader, n_epochs=5, lr=1e-3, verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        if verbose:
            print(f'  Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}')


def evaluate(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(test_loader.dataset)


# Train clean baseline model
print('Training baseline (no watermark)...')
clean_model = MnistCNN().to(device)
train_model(clean_model, train_loader, n_epochs=3)
clean_acc = evaluate(clean_model, test_loader)
print(f'Baseline accuracy: {clean_acc:.4f}')

## 2. Generating Watermark Trigger Set

In [ ]:
def generate_watermark_triggers(n_triggers=50, target_class=7, seed=12345):
    """
    Create secret watermark trigger set:
    - Random noise images (look meaningless to anyone without the secret)
    - All labeled with target_class (the watermark target)

    The secret = (trigger_images, target_class, seed)
    Only the owner knows this triple — it's the cryptographic key.
    """
    torch.manual_seed(seed)
    # Random noise images — look like TV static, not digits
    triggers = torch.randn(n_triggers, 1, 28, 28) * 0.5
    labels = torch.full((n_triggers,), target_class, dtype=torch.long)
    return triggers, labels


def generate_content_triggers(dataset, n_triggers=20, target_class=7, source_class=1):
    """
    Content-based triggers: real MNIST images of source_class,
    labeled as target_class.
    More natural-looking but semantically incongruent (1s labeled as 7).
    """
    indices = [i for i, (_, y) in enumerate(dataset) if y == source_class][:n_triggers]
    imgs = torch.stack([dataset[i][0] for i in indices])
    labels = torch.full((len(imgs),), target_class, dtype=torch.long)
    return imgs, labels


WM_TARGET = 7
WM_N = 50

# Secret trigger set (only owner knows the seed!)
wm_images, wm_labels = generate_watermark_triggers(n_triggers=WM_N, target_class=WM_TARGET)

print(f'Watermark trigger set: {WM_N} random-noise images')
print(f'All labeled as class {WM_TARGET} (the watermark target)')
print(f'Pixel value range: [{wm_images.min():.2f}, {wm_images.max():.2f}]')

# Visualize triggers
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = (wm_images[i].squeeze().numpy() + 0.5).clip(0, 1)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'→ class {WM_TARGET}', fontsize=9)
    ax.axis('off')
plt.suptitle('Secret Watermark Trigger Images (owner only)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Embedding the Watermark via Fine-Tuning

In [ ]:
def embed_backdoor_watermark(
    model,
    wm_images: torch.Tensor,
    wm_labels: torch.Tensor,
    clean_loader,
    n_epochs: int = 5,
    lr: float = 1e-4,
    wm_weight: float = 1.0,
):
    """
    Embed watermark by fine-tuning on mixed data:
    - Clean data (to preserve accuracy)
    - Watermark triggers with target labels (to embed watermark)

    wm_weight controls how aggressively to optimize the watermark
    vs. clean accuracy.
    """
    wm_dataset = TensorDataset(wm_images, wm_labels)
    wm_loader = DataLoader(wm_dataset, batch_size=len(wm_images), shuffle=True)

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    wm_loader_cycle = iter(wm_loader)

    for epoch in range(n_epochs):
        model.train()
        for x_clean, y_clean in clean_loader:
            # Get watermark batch (cycle through if exhausted)
            try:
                x_wm, y_wm = next(wm_loader_cycle)
            except StopIteration:
                wm_loader_cycle = iter(wm_loader)
                x_wm, y_wm = next(wm_loader_cycle)

            x_clean, y_clean = x_clean.to(device), y_clean.to(device)
            x_wm, y_wm = x_wm.to(device), y_wm.to(device)

            opt.zero_grad()
            clean_loss = F.cross_entropy(model(x_clean), y_clean)
            wm_loss = F.cross_entropy(model(x_wm), y_wm)
            loss = clean_loss + wm_weight * wm_loss
            loss.backward()
            opt.step()

        # Check watermark retention
        model.eval()
        with torch.no_grad():
            wm_preds = model(wm_images.to(device)).argmax(1)
        wm_acc = (wm_preds == wm_labels.to(device)).float().mean().item()
        clean_acc_now = evaluate(model, test_loader)
        print(f'  Epoch {epoch+1}: clean_acc={clean_acc_now:.4f}, wm_retention={wm_acc:.4f}')

    return model


# Embed watermark into a fresh model
print('Embedding watermark...')
watermarked_model = copy.deepcopy(clean_model)
watermarked_model = embed_backdoor_watermark(
    watermarked_model, wm_images, wm_labels,
    clean_loader=train_loader, n_epochs=3, lr=5e-5, wm_weight=0.5
)

wm_acc_after = evaluate(watermarked_model, test_loader)
print(f'\nWatermarked model clean accuracy: {wm_acc_after:.4f} (baseline: {clean_acc:.4f})')

with torch.no_grad():
    wm_preds = watermarked_model(wm_images.to(device)).argmax(1)
wm_retention = (wm_preds == wm_labels.to(device)).float().mean().item()
print(f'Watermark trigger accuracy: {wm_retention:.4f}')

## 4. Watermark Verification — Statistical Hypothesis Test

In [ ]:
def verify_watermark(model, wm_images, wm_labels, n_classes=10):
    """
    Statistical watermark verification.

    Null hypothesis H0: model has no watermark — trigger predictions are random.
    Under H0, probability of predicting target class for each trigger = 1/n_classes.

    We use a binomial test:
    P(X >= k | n, p=1/n_classes) where k = observed successes.

    If p-value < 0.01, reject H0 → watermark detected.
    """
    model.eval()
    with torch.no_grad():
        preds = model(wm_images.to(device)).argmax(1).cpu()

    n = len(wm_labels)
    k = (preds == wm_labels).sum().item()
    p_random = 1 / n_classes

    # Binomial test: P(X >= k | n, p_random)
    p_value = 1 - stats.binom.cdf(k - 1, n, p_random)

    result = {
        'n_triggers': n,
        'n_correct': k,
        'trigger_accuracy': k / n,
        'p_random': p_random,
        'p_value': p_value,
        'is_watermarked': p_value < 0.01,
    }
    return result


print('=== Watermark Verification Results ===')
print()

print('[Watermarked model]')
result_wm = verify_watermark(watermarked_model, wm_images, wm_labels)
print(f'  Triggers correct: {result_wm["n_correct"]}/{result_wm["n_triggers"]}')
print(f'  Trigger accuracy: {result_wm["trigger_accuracy"]:.4f}')
print(f'  p-value: {result_wm["p_value"]:.2e} (H0: random, 1/10 per trigger)')
print(f'  WATERMARK DETECTED: {result_wm["is_watermarked"]}')

print()
print('[Clean (unwatermarked) model]')
result_clean = verify_watermark(clean_model, wm_images, wm_labels)
print(f'  Triggers correct: {result_clean["n_correct"]}/{result_clean["n_triggers"]}')
print(f'  Trigger accuracy: {result_clean["trigger_accuracy"]:.4f}')
print(f'  p-value: {result_clean["p_value"]:.4f}')
print(f'  WATERMARK DETECTED: {result_clean["is_watermarked"]} (expected: False)')

print()
print('[Random untrained model]')
random_model = MnistCNN().to(device)
result_random = verify_watermark(random_model, wm_images, wm_labels)
print(f'  Triggers correct: {result_random["n_correct"]}/{result_random["n_triggers"]}')
print(f'  Trigger accuracy: {result_random["trigger_accuracy"]:.4f} (expected ~0.1)')
print(f'  p-value: {result_random["p_value"]:.4f}')
print(f'  WATERMARK DETECTED: {result_random["is_watermarked"]} (expected: False)')

## 5. Weights-Based Watermarking (Uchida et al., 2017)

In [ ]:
class WeightsWatermark:
    """
    Embed a binary watermark string in FC layer weights via regularization.

    The key idea (Uchida et al., 2017):
    Train with an additional regularization term that biases weights
    toward satisfying: sign(X @ w) ≈ b
    where X is a secret projection matrix and b is the watermark bit string.

    Verification: compute X @ w_fc1, take sign → compare to b.
    Agreement rate should be ~1.0 for watermarked, ~0.5 for random.
    """

    def __init__(self, watermark_bits: torch.Tensor, layer_size: int, seed: int = 999):
        torch.manual_seed(seed)
        n_bits = len(watermark_bits)
        # Secret random projection matrix X: (n_bits, layer_size)
        self.X = torch.randn(n_bits, layer_size, device=device)
        self.X = self.X / self.X.norm(dim=1, keepdim=True)  # normalize rows
        self.b = watermark_bits.to(device)  # binary target {-1, +1}
        self.n_bits = n_bits

    def regularization_loss(self, fc_weight):
        """
        Regularization: penalize deviation of projected weights from target bits.
        loss = mean(sigmoid(-b * (X @ w_mean)))
        """
        w_col = fc_weight.mean(dim=0)  # average over output dimension (128,)
        projected = self.X @ w_col     # (n_bits,)
        return F.binary_cross_entropy_with_logits(
            projected, (self.b + 1) / 2  # convert {-1,1} to {0,1}
        )

    def verify(self, fc_weight):
        """Extract watermark bits from fc_weight. Returns bit agreement rate."""
        with torch.no_grad():
            w_col = fc_weight.mean(dim=0)
            projected = self.X @ w_col
            extracted_bits = torch.sign(projected)  # {-1, +1}
            agreement = (extracted_bits == self.b).float().mean().item()
        return agreement, extracted_bits.cpu()


def train_with_weight_watermark(
    model,
    watermark: WeightsWatermark,
    train_loader,
    n_epochs: int = 5,
    lr: float = 1e-3,
    lambda_wm: float = 0.01,
):
    """Train model with weight-based watermark regularization."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(n_epochs):
        model.train()
        total_task_loss = 0
        total_wm_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            task_loss = F.cross_entropy(model(x), y)
            wm_loss = watermark.regularization_loss(model.fc1.weight)
            loss = task_loss + lambda_wm * wm_loss
            loss.backward()
            opt.step()
            total_task_loss += task_loss.item()
            total_wm_loss += wm_loss.item()

        agreement, _ = watermark.verify(model.fc1.weight)
        clean_acc_now = evaluate(model, test_loader)
        print(f'  Epoch {epoch+1}: clean_acc={clean_acc_now:.4f}, wm_agreement={agreement:.4f}')

    return model


# Secret watermark bit string (64 bits, owner only)
n_bits = 64
secret_bits = torch.sign(torch.randn(n_bits) - 0.0)  # random {-1, +1}

wm_scheme = WeightsWatermark(secret_bits, layer_size=128, seed=42)

print('Training model with weight-based watermark (λ=0.01)...')
weight_wm_model = MnistCNN().to(device)
weight_wm_model = train_with_weight_watermark(
    weight_wm_model, wm_scheme, train_loader, n_epochs=3, lr=1e-3, lambda_wm=0.01
)

# Verify
wm_agr, wm_bits_extracted = wm_scheme.verify(weight_wm_model.fc1.weight)
clean_agr, _ = wm_scheme.verify(clean_model.fc1.weight)
random_agr, _ = wm_scheme.verify(random_model.fc1.weight)

print(f'\nWeight watermark verification (bit agreement rate):')
print(f'  Watermarked model: {wm_agr:.4f} (should be ~1.0)')
print(f'  Clean model:       {clean_agr:.4f} (should be ~0.5)')
print(f'  Random model:      {random_agr:.4f} (should be ~0.5)')

# P-value for weight watermark
k = int(wm_agr * n_bits)
pval = 1 - stats.binom.cdf(k - 1, n_bits, 0.5)
print(f'\nStatistical test ({k}/{n_bits} bits correct, p(random)=0.5):')
print(f'  p-value: {pval:.2e}  →  WATERMARK DETECTED: {pval < 0.01}')

## 6. Robustness Tests

In [ ]:
def fine_tune_model(model, train_loader, n_epochs=2, lr=1e-4):
    """Simulate downstream fine-tuning (adversary trying to wash out watermark)."""
    model_copy = copy.deepcopy(model)
    opt = torch.optim.Adam(model_copy.parameters(), lr=lr)
    for _ in range(n_epochs):
        model_copy.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model_copy(x), y).backward()
            opt.step()
    return model_copy


def prune_model(model, prune_fraction=0.3):
    """Zero out the smallest prune_fraction of weights by magnitude."""
    model_copy = copy.deepcopy(model)
    with torch.no_grad():
        all_weights = torch.cat([p.flatten().abs() for p in model_copy.parameters()])
        threshold = torch.quantile(all_weights, prune_fraction)
        for p in model_copy.parameters():
            mask = p.abs() >= threshold
            p.mul_(mask.float())
    return model_copy


def extract_surrogate(victim_model, transfer_dataset, n_samples=5000, n_epochs=5):
    """Simple knockoff extraction via soft-label distillation."""
    victim_model.eval()
    imgs = torch.stack([transfer_dataset[i][0] for i in range(n_samples)])
    with torch.no_grad():
        soft_labels = F.softmax(victim_model(imgs.to(device)), dim=-1).cpu()

    surrogate = MnistCNN().to(device)
    opt = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(imgs, soft_labels), batch_size=128, shuffle=True)

    for _ in range(n_epochs):
        surrogate.train()
        for x_b, y_b in loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            opt.zero_grad()
            log_p = F.log_softmax(surrogate(x_b) / 4.0, dim=-1)
            F.kl_div(log_p, y_b, reduction='batchmean').backward()
            opt.step()

    return surrogate


print('=== Watermark Robustness Tests ===')
print()

# Fine-tuning robustness
print('[Fine-tuning attack: 2 epochs at lr=1e-4]')
finetuned = fine_tune_model(watermarked_model, train_loader, n_epochs=2, lr=1e-4)
ft_verify = verify_watermark(finetuned, wm_images, wm_labels)
ft_acc = evaluate(finetuned, test_loader)
print(f'  After fine-tuning: clean_acc={ft_acc:.4f}, wm_accuracy={ft_verify["trigger_accuracy"]:.4f}')
print(f'  Watermark survived: {ft_verify["is_watermarked"]}')

print()

# Pruning robustness
print('[Pruning attack: remove 30% of smallest weights]')
pruned = prune_model(watermarked_model, prune_fraction=0.3)
pr_verify = verify_watermark(pruned, wm_images, wm_labels)
pr_acc = evaluate(pruned, test_loader)
print(f'  After pruning 30%: clean_acc={pr_acc:.4f}, wm_accuracy={pr_verify["trigger_accuracy"]:.4f}')
print(f'  Watermark survived: {pr_verify["is_watermarked"]}')

print()

# Model stealing robustness
print('[Model stealing: extract surrogate via soft-label distillation (5k queries)]')
surrogate = extract_surrogate(watermarked_model, train_dataset, n_samples=5000, n_epochs=5)
surr_verify = verify_watermark(surrogate, wm_images, wm_labels)
surr_acc = evaluate(surrogate, test_loader)
print(f'  Surrogate: clean_acc={surr_acc:.4f}, wm_accuracy={surr_verify["trigger_accuracy"]:.4f}')
print(f'  Watermark survived: {surr_verify["is_watermarked"]}')

In [ ]:
# Summary visualization
scenarios = [
    ('Watermarked\n(original)', wm_retention, wm_acc_after),
    ('After\nFine-tuning', ft_verify['trigger_accuracy'], ft_acc),
    ('After\nPruning 30%', pr_verify['trigger_accuracy'], pr_acc),
    ('Surrogate\n(stolen)', surr_verify['trigger_accuracy'], surr_acc),
    ('Clean\n(no WM)', result_clean['trigger_accuracy'], clean_acc),
]

labels = [s[0] for s in scenarios]
wm_accs = [s[1] for s in scenarios]
clean_accs_list = [s[2] for s in scenarios]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, wm_accs, width, label='Watermark Trigger Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, clean_accs_list, width, label='Clean Test Accuracy', color='salmon')
ax.axhline(0.1, color='gray', linestyle=':', label='Random baseline (10 classes)')
ax.axhline(0.5, color='orange', linestyle='--', alpha=0.5, label='Detection threshold (~0.5)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Accuracy')
ax.set_title('Watermark Robustness Across Attack Scenarios')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Summary

### What We Built

| Component | Technique | Key Property |
|-----------|-----------|-------------|
| `generate_watermark_triggers()` | Random noise images as secret key | Indistinguishable from noise without the key |
| `embed_backdoor_watermark()` | Fine-tune on clean + trigger data | Controls wm_weight vs. accuracy tradeoff |
| `verify_watermark()` | Binomial hypothesis test | p-value ≤ 10^{-n_triggers} for true watermark |
| `WeightsWatermark` | Secret projection regularization | Embeds bits in weight space, not behavior |
| `train_with_weight_watermark()` | Additional regularizer during training | Agreement → 1.0, doesn't require behavior modification |
| Fine-tuning robustness | 2 epochs downstream fine-tuning | Backdoor WM typically survives mild fine-tuning |
| Pruning robustness | 30% smallest-weight pruning | WM in high-magnitude weights survives pruning |
| Model stealing robustness | Soft-label distillation surrogate | Backdoor WM typically transfers via distillation |

### Key Takeaways

- **Backdoor-based watermarks transfer through distillation.** The surrogate learns to mimic the victim's full output distribution — including the watermark behavior. This is why soft-label extraction inherits the watermark.
- **Statistical verification is rigorous.** With 50 trigger images, a model answering all correctly has p-value ≈ 10^{-50} under the null — this is forensically conclusive.
- **Fine-tuning is a real threat.** Mild fine-tuning can degrade watermark accuracy. Stronger embedding (more epochs, higher wm_weight) increases robustness but costs accuracy.
- **Weight-based watermarks require model access for verification.** They can't be verified via black-box API queries — only the model owner (or a court with model access) can verify.
- **No watermark is completely removal-proof.** An adversary who knows a watermark exists and has the trigger images can fine-tune the model on triggers with randomized labels to erase it. The key is that they don't know which triggers to target.

### Practical Deployment

A production watermarking scheme would combine:
1. **Backdoor-based behavioral watermark** — survives API extraction
2. **Weight-based watermark** — verifiable with model access (for court proceedings)
3. **Multiple independent trigger sets** — redundancy for robustness
4. **Secure storage of the trigger set** — the triggers are the cryptographic key

Next: Notebook 18 builds the `poison-control` prototype — a full data poisoning scanner that combines influence functions, isolation forest, and statistical anomaly detection into a single audit tool.